In [3]:
import pandas as pd
import os
import zipfile
import numpy as np
import torch
from torch.utils.data import Dataset
import timm
import torchvision.transforms as T
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import mlflow

d:\kaggle_jaguar\jaguar_kaggle\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
batch_size = 8

In [ ]:
DATASET_PATH = "jaguar-re-id.zip"
EXTRACT_PATH = "data"

if not os.path.exists(EXTRACT_PATH):
    print("Разархивирование датасета...")
    with zipfile.ZipFile(DATASET_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_PATH)
    print(f"Датасет разархивирован в {EXTRACT_PATH}")
else:
    print(f"Датасет уже находится в {EXTRACT_PATH}")

Датасет уже находится в data


In [ ]:
path_train = 'data/train.csv'
kaggle_train_path = '/kaggle/input/competitions/jaguar-re-id/train.csv'
kaggle_test_path = '/kaggle/input/competitions/jaguar-re-id/test.csv'
path_test = 'data/test.csv'
train = pd.read_csv(kaggle_train_path)
test = pd.read_csv(kaggle_test_path)

In [ ]:
encoder = LabelEncoder()

In [ ]:
train['ground_truth'] = encoder.fit_transform(train['ground_truth'])

In [ ]:
train.head()

,filename,ground_truth
0,train_0001.png,0
1,train_0002.png,0
2,train_0003.png,0
3,train_0004.png,1
4,train_0005.png,1


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    train['filename'], train['ground_truth'], test_size=0.15, random_state=12)

In [ ]:
transform = T.Compose([T.Resize(size=(384, 384)),
                              T.ToTensor(), 
                              T.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])])

In [ ]:
class CustomDataset(Dataset):
    def __init__(self, X=X_train, y=y_train, img_dir='/kaggle/input/competitions/jaguar-re-id/train/train'):
        self.X = X
        self.y = y
        self.img = img_dir
        self.transform = T.Compose([T.Resize(size=(384, 384)),
                              T.ToTensor(), 
                              T.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])]) 
         
    def __len__(self):
        return self.X.shape[0]
    
    def __getitem__(self, idx):
        img_path = self.X.iloc[idx]
        img = Image.open(self.img + '/' + img_path).convert('RGB')
        return {'x': self.transform(img), 'y': self.y.iloc[idx]}

In [ ]:
dataset_1 = CustomDataset()

In [ ]:
dataset_test = CustomDataset(X_test, y_test)

In [ ]:
dataset_1.__len__()

1610

In [ ]:
dataset_1.__getitem__(idx=1123)

{'x': tensor([[[ 0.2863,  0.3176,  0.3725,  ..., -0.1294, -0.1059, -0.0824],
          [ 0.0353,  0.1216,  0.2235,  ..., -0.2078, -0.1765, -0.0431],
          [-0.0980, -0.0824, -0.0353,  ..., -0.3255, -0.1765,  0.0039],
          ...,
          [ 0.1686,  0.0902,  0.0667,  ..., -0.2549, -0.2078, -0.2078],
          [ 0.1529,  0.1294,  0.1373,  ..., -0.2863, -0.1843, -0.1529],
          [ 0.1373,  0.1216,  0.1294,  ..., -0.2392, -0.1922, -0.1843]],
 
         [[ 0.2078,  0.2392,  0.2941,  ..., -0.2078, -0.1843, -0.1608],
          [-0.0667,  0.0118,  0.1137,  ..., -0.2863, -0.2549, -0.1216],
          [-0.2314, -0.2078, -0.1608,  ..., -0.4039, -0.2549, -0.0824],
          ...,
          [ 0.1373,  0.0588,  0.0353,  ..., -0.3412, -0.2941, -0.2941],
          [ 0.1137,  0.0902,  0.0980,  ..., -0.3725, -0.2706, -0.2392],
          [ 0.1059,  0.0824,  0.0902,  ..., -0.3255, -0.2784, -0.2706]],
 
         [[ 0.1373,  0.1843,  0.2627,  ..., -0.2235, -0.2000, -0.1765],
          [-0.1529, -0.

In [ ]:
from transformers import DefaultDataCollator

In [ ]:
data_collator = DefaultDataCollator()

In [ ]:
train_dataloader = DataLoader(dataset_1, batch_size=batch_size, shuffle=True, collate_fn=data_collator)

In [ ]:
test_dataloader = DataLoader(dataset_test, batch_size=batch_size, shuffle=True, collate_fn=data_collator)

In [ ]:
model = timm.create_model("hf-hub:BVRA/MegaDescriptor-L-384", pretrained=True, num_classes=31)

In [ ]:
def compute_metrics(eval_pred, label):
    predictions = eval_pred
    predictions = np.argmax(predictions, axis=1)
    
    accuracy = accuracy_score(label, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(label, predictions, average='weighted')
    
    return {
        'accuracy': accuracy,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

In [ ]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=5e-5)

In [ ]:
from transformers import get_scheduler

In [ ]:
num_epochs = 5
num_training_steps = num_epochs * len(train_dataloader)
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)
print(num_training_steps)

1010


In [ ]:
import torch.nn as nn
criterion = nn.CrossEntropyLoss()

In [ ]:
torch.cuda.empty_cache()

In [ ]:
from tqdm.auto import tqdm

progress_bar = tqdm(range(num_training_steps))
best_loss = 1000
model.to('cuda')
for epoch in range(num_epochs):
    model.train()
    for batch in train_dataloader:
        data = batch
        X = data['x'].to('cuda')
        y = data['y'].to('cuda')
        outputs = model(X)
        loss = criterion(outputs, y)
        loss.backward()

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update()
        print(loss.item())
        
    model.eval()
    metrics_to_eval = []
    count = 1
    for batch in test_dataloader:
        data = batch
        X = data['x'].to('cuda')
        y = data['y'].to('cuda')
        outputs = model(X)
        loss = criterion(outputs, y)
        if loss < best_loss:
            model.save(model.state_dict(), '/kaggle/working/re_identification_v1')
        metrics_to_eval.append([count, compute_metrics(outputs, y)])
        count += 1
        progress_bar.update()
        print(loss.item())
    print(metrics_to_eval)
    torch.cuda.empty_cache()